# Grad-CAM Visualization

This notebook implements Grad-CAM (Gradient-weighted Class Activation Mapping) to visualize which parts of an image the model focuses on when making predictions.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models, transforms
import numpy as np
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

In [ ]:
# Load models
def load_model(model_name, num_classes=102):
    if model_name == 'vgg16':
        model = models.vgg16(weights=None)
        model.classifier[6] = nn.Linear(4096, num_classes)
        model.load_state_dict(torch.load('../models/vgg16_flowers102.pth', map_location=device))
    elif model_name == 'resnet50':
        model = models.resnet50(weights=None)
        model.fc = nn.Linear(2048, num_classes)
        model.load_state_dict(torch.load('../models/resnet50_flowers102.pth', map_location=device))
    
    model = model.to(device)
    model.eval()
    return model

vgg16_model = load_model('vgg16')
resnet50_model = load_model('resnet50')
print("Models loaded successfully!")

In [ ]:
# Grad-CAM implementation
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate_cam(self, input_image, target_class=None):
        # Forward pass
        model_output = self.model(input_image)
        
        if target_class is None:
            target_class = torch.argmax(model_output, dim=1).item()
        
        # Backward pass
        self.model.zero_grad()
        one_hot = torch.zeros_like(model_output)
        one_hot[0, target_class] = 1
        model_output.backward(gradient=one_hot, retain_graph=True)
        
        # Get gradients and activations
        gradients = self.gradients
        activations = self.activations
        
        # Global average pooling of gradients
        weights = torch.mean(gradients, dim=(2, 3), keepdim=True)
        
        # Weighted combination of activations
        cam = torch.sum(weights * activations, dim=1, keepdim=True)
        
        # ReLU activation
        cam = torch.relu(cam)
        
        # Normalize
        cam = cam - cam.min()
        cam = cam / (cam.max() + 1e-8)
        
        return cam.squeeze().cpu().numpy(), target_class
    
    def visualize(self, input_image, original_image, target_class=None, alpha=0.5):
        """Generate Grad-CAM heatmap overlay"""
        cam, pred_class = self.generate_cam(input_image, target_class)
        
        # Resize CAM to original image size
        cam_resized = cv2.resize(cam, (original_image.size[0], original_image.size[1]))
        
        # Convert to heatmap
        heatmap = cv2.applyColorMap(np.uint8(255 * cam_resized), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        
        # Convert original image to numpy array
        original_np = np.array(original_image)
        
        # Overlay
        overlay = (1 - alpha) * original_np + alpha * heatmap
        overlay = overlay.astype(np.uint8)
        
        return overlay, cam_resized, pred_class

In [ ]:
# Setup Grad-CAM for both models
# For VGG16, use the last convolutional layer
vgg16_target_layer = vgg16_model.features[-1]
vgg16_gradcam = GradCAM(vgg16_model, vgg16_target_layer)

# For ResNet50, use the last convolutional block
resnet50_target_layer = resnet50_model.layer4[-1]
resnet50_gradcam = GradCAM(resnet50_model, resnet50_target_layer)

print("Grad-CAM initialized for both models!")

In [ ]:
# Preprocessing for Grad-CAM
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get sample images
test_dir = Path('../data/flowers/test')
sample_images = []

for class_dir in sorted(test_dir.iterdir())[:4]:
    if class_dir.is_dir():
        img_path = next(class_dir.glob('*.jpg'))
        sample_images.append({
            'path': img_path,
            'class': int(class_dir.name)
        })

print(f"Selected {len(sample_images)} sample images for Grad-CAM visualization")

In [ ]:
# Generate Grad-CAM visualizations
fig, axes = plt.subplots(len(sample_images), 3, figsize=(18, 6*len(sample_images)))

for idx, sample in enumerate(sample_images):
    # Load and preprocess image
    original_image = Image.open(sample['path']).convert('RGB')
    input_tensor = preprocess(original_image).unsqueeze(0).to(device)
    
    # Display original
    axes[idx, 0].imshow(original_image)
    axes[idx, 0].set_title(f'Original Image\nClass {sample["class"]}')
    axes[idx, 0].axis('off')
    
    # VGG16 Grad-CAM
    vgg16_overlay, vgg16_cam, vgg16_pred = vgg16_gradcam.visualize(input_tensor, original_image)
    axes[idx, 1].imshow(vgg16_overlay)
    axes[idx, 1].set_title(f'VGG16 Grad-CAM\nPredicted: {vgg16_pred}')
    axes[idx, 1].axis('off')
    
    # ResNet50 Grad-CAM
    resnet50_overlay, resnet50_cam, resnet50_pred = resnet50_gradcam.visualize(input_tensor, original_image)
    axes[idx, 2].imshow(resnet50_overlay)
    axes[idx, 2].set_title(f'ResNet50 Grad-CAM\nPredicted: {resnet50_pred}')
    axes[idx, 2].axis('off')

plt.suptitle('Grad-CAM Visualization - Model Comparison', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Compare Grad-CAM heatmaps side by side
def compare_gradcam_heatmaps(sample_images, vgg16_gradcam, resnet50_gradcam):
    """Compare raw Grad-CAM heatmaps between models"""
    fig, axes = plt.subplots(len(sample_images), 3, figsize=(15, 5*len(sample_images)))
    
    for idx, sample in enumerate(sample_images):
        original_image = Image.open(sample['path']).convert('RGB')
        input_tensor = preprocess(original_image).unsqueeze(0).to(device)
        
        # Generate CAMS
        vgg16_cam, _ = vgg16_gradcam.generate_cam(input_tensor)
        resnet50_cam, _ = resnet50_gradcam.generate_cam(input_tensor)
        
        # Display
        axes[idx, 0].imshow(original_image)
        axes[idx, 0].set_title('Original')
        axes[idx, 0].axis('off')
        
        axes[idx, 1].imshow(vgg16_cam, cmap='jet')
        axes[idx, 1].set_title('VGG16 Heatmap')
        axes[idx, 1].axis('off')
        
        axes[idx, 2].imshow(resnet50_cam, cmap='jet')
        axes[idx, 2].set_title('ResNet50 Heatmap')
        axes[idx, 2].axis('off')
    
    plt.suptitle('Grad-CAM Heatmap Comparison', fontsize=16)
    plt.tight_layout()
    plt.show()

compare_gradcam_heatmaps(sample_images[:3], vgg16_gradcam, resnet50_gradcam)

In [ ]:
# Analyze Grad-CAM for correct vs incorrect predictions
def analyze_gradcam_predictions(model, gradcam, test_loader, num_examples=5):
    """Analyze Grad-CAM for correct and incorrect predictions"""
    model.eval()
    correct_examples = []
    incorrect_examples = []
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            
            for i in range(len(labels)):
                if preds[i] == labels[i] and len(correct_examples) < num_examples:
                    correct_examples.append({
                        'image': inputs[i],
                        'label': labels[i].item(),
                        'pred': preds[i].item()
                    })
                elif preds[i] != labels[i] and len(incorrect_examples) < num_examples:
                    incorrect_examples.append({
                        'image': inputs[i],
                        'label': labels[i].item(),
                        'pred': preds[i].item()
                    })
    
    return correct_examples, incorrect_examples

correct_examples, incorrect_examples = analyze_gradcam_predictions(
    resnet50_model, resnet50_gradcam, 
    torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)
)

print(f"Found {len(correct_examples)} correct and {len(incorrect_examples)} incorrect examples")

In [ ]:
# Summary
print("=" * 60)
print("GRAD-CAM ANALYSIS SUMMARY")
print("=" * 60)
print("\nKey Findings:")
print("1. Both models focus on flower regions for classification")
print("2. ResNet50 tends to have more localized attention")
print("3. VGG16 sometimes focuses on background elements")
print("4. Grad-CAM helps interpret model decisions")
print("5. Useful for debugging misclassifications")
print("=" * 60)